# Figure 2 - dataset overview

Combined dataset-overview figure (panels a-e). Reads trimmed tables from `data/training_overview/` and writes `figures/fig2/dataset_overview.{svg,png}`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Self-contained repo paths (no absolute filesystem paths in this notebook).
PUB = Path.cwd().parent
sys.path.insert(0, str(PUB / "lib"))
DATA, RESULTS, FIGURES = PUB / "data", PUB / "results", PUB / "figures"

In [ ]:
# Vendored figure styling -- single source of truth for the figure.
# apply_style() sets editable-SVG text, white background, 350 dpi, and >=12 pt fonts.
from figure_style import apply_style, figsize, set_y_headroom, add_panel_labels, save_figure

apply_style()

# Warm accent colour + `charizard` cmap for the MHC donut.
from pypalettes import load_cmap

color = "#D86020FF"
cmap = load_cmap("charizard")

## Load training data

In [ ]:
# Positive training rows (binder == 1), trimmed to the columns this figure needs.
pos_training = pd.read_table(DATA / "training_overview" / "pos_training.tsv")

# 4-digit (2-field) MHC typing.
pos_training["mhc"] = pos_training["mhc"].str.split(":").str[0:2].str.join(":")

# Attach epitope species via the identifier -> species lookup.
species = pd.read_table(DATA / "training_overview" / "species_lookup.tsv")
pos_training_species = pos_training.merge(species, on="identifier", how="left")

# One peptide (identifier df356728-...) is absent from the species lookup; label it.
pos_training_species.loc[
    pos_training_species["identifier"] == "df356728-d291-4e0f-ba1f-e5787941fdce",
    "Epitope species",
] = "Homo Sapiens"

# Consolidate / prettify species names.
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Humanimmunodeficiencyvirus1(humanimmunodeficiencyvirus1HIV-1)", "HIV-1", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Humanherpesvirus4(EpsteinBarrvirus)", "EBV", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Humanherpesvirus5(Humancytomegalovirus)", "EBV", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Vacciniavirus(vacciniavirusVV)", "VV", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Merkelcellpolyomavirus(MCPyVisolateR17b)", "MCPyV", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Merkelcellpolyomavirus(MCPyVisolateR17b)", "MCPyV", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "HumanT-cellleukemiavirustypeI(HumanTcellleukemiavirustype1)", "HTLV-1", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "InfluenzaAvirus", "InfluenzaA", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Influenza", "InfluenzaA", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "SARS-CoV2", "SARS-CoV-2", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "InfluenzaA", "Influenza A", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "HomoSapiens", "Homo Sapiens", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "InfluenzaB", "Influenza B", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "Homosapiens(human)", "Homo Sapiens", pos_training_species["Epitope species"])
pos_training_species["Epitope species"] = np.where(pos_training_species["Epitope species"] == "DiabetesType1", "T1 Diabetes", pos_training_species["Epitope species"])

pos_training_species["Epitope species"].value_counts()

In [ ]:
# Inline sanity checks on the loaded/derived frames.
assert len(pos_training) == 20809, len(pos_training)
assert pos_training["peptide"].nunique() == 77, pos_training["peptide"].nunique()
assert pos_training["mhc"].nunique() == 14, pos_training["mhc"].nunique()
assert len(pos_training_species) == len(pos_training)
assert pos_training_species["Epitope species"].isna().sum() == 0
print("assertions passed:", len(pos_training), "positive training rows")

## Figure

In [ ]:
# ============================================================================
# Combined dataset-overview figure (panels a-e):
#   a peptide counts | b MHC donut | c epitope species
#   d PAE boxplot-by-peptide | e pLDDT boxplot-by-peptide
# Uses pos_training / pos_training_species / color / cmap from the cells above.
# Dense categorical labels (77 peptides, 16 species) stay small so they read as a group.
# ============================================================================
PAE_COL, PLDDT_COL = "model_2_ptm_pae", "model_2_ptm_plddt"


def _panel_a_peptides(ax, pos):
    counts = (pos.groupby("peptide").size().reset_index(name="count")
              .sort_values("count", ascending=True))
    ax.barh(counts["peptide"], counts["count"],
            alpha=0.8, edgecolor="white", linewidth=0.5, color=color)
    ax.set_xlabel("Count", fontsize=12)
    ax.set_ylabel("Peptide", fontsize=12)
    ax.set_xscale("log")
    ax.set_xticks([1,100,10000])
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x)}"))
    ax.tick_params(axis="y", labelsize=5.5)
    ax.tick_params(axis="x", labelsize=12)
    ax.set_axisbelow(True); ax.margins(y=0.01)
    ax.set_xlim(0.7,40000)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def _panel_b_donut(ax, pos):
    hla = (pos.groupby("mhc").size().reset_index(name="count")
           .sort_values("count", ascending=False))
    colors = [cmap(i / len(hla)) for i in range(len(hla))]
    wedges, texts, autotexts = ax.pie(
        hla["count"], labels=hla["mhc"], colors=colors,
        autopct="%1.1f%%", startangle=90, pctdistance=0.82,
        wedgeprops=dict(width=0.5, edgecolor="white", linewidth=1.5))
    ax.add_artist(plt.Circle((0, 0), 0.70, fc="white"))
    pct = hla["count"].to_numpy() / hla["count"].sum() * 100
    for i, (t, at) in enumerate(zip(texts, autotexts)):
        if pct[i] < 2.0:
            t.set_text(""); at.set_text("")
        else:
            t.set_fontsize(7)                              # smaller allele labels
            if pct[i] > 5.0:                                # only show % for wedges > 5 %
                at.set_color("black"); at.set_fontsize(6)
            else:
                at.set_text("")
    ax.text(0, 0, f"MHC Types\n({len(pos):,} TCRs)",
            ha="center", va="center", fontsize=10)
    ax.axis("equal")


def _panel_c_species(ax, spc):
    counts = (spc.groupby("Epitope species").size().reset_index(name="count")
              .sort_values("count", ascending=False))
    bars = ax.bar(counts["Epitope species"], counts["count"],
                  alpha=0.8, edgecolor="white", linewidth=0.5, color=color)
    ax.set_xlabel("Epitope Species", fontsize=12)
    ax.set_ylabel("TCR-pMHC Pairs", fontsize=12)
    ax.set_yticks(np.arange(0, 13000, 2000))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x):,}"))
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts["Epitope species"], rotation=45, ha="right", fontsize=8)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_axisbelow(True); ax.margins(x=0.03)
    # value labels: stagger the crowded short bars vertically so neighbours don't collide
    ymax = counts["count"].max()
    for i, (bar, c) in enumerate(zip(bars, counts["count"])):
        small = c < 1000
        off = ymax * (0.015 + (i % 2) * 0.055) if small else ymax * 0.012
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + off,
                f"{c:,}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, ymax * 1.16)   # headroom for the raised labels
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def _panels_de_boxplots(ax_pae, ax_plddt, pos):
    order = (pos.groupby("peptide")[PAE_COL].median()
             .sort_values(ascending=True).index)
    for ax, col, ylabel in [(ax_pae, PAE_COL, "PAE (Å)"),
                            (ax_plddt, PLDDT_COL, "pLDDT")]:
        sns.boxplot(data=pos, x="peptide", y=col, order=order, ax=ax,
                    color=color, width=0.65, linewidth=0.7, fliersize=0,
                    boxprops=dict(alpha=0.85, edgecolor="#333333"),
                    medianprops=dict(color="#333333", linewidth=1.2),
                    whiskerprops=dict(color="#333333", linewidth=0.7),
                    capprops=dict(color="#333333", linewidth=0.7))
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xlabel("")
        ax.tick_params(axis="y", labelsize=12)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        ax.spines["left"].set_linewidth(1.2)
        ax.spines["bottom"].set_linewidth(1.2)
    ax_pae.tick_params(axis="x", labelbottom=False)
    ax_plddt.set_xlabel("Peptide", fontsize=12)
    plt.setp(ax_plddt.get_xticklabels(), rotation=90, ha="center", fontsize=5.5)


fig = plt.figure(figsize=figsize("double", 270), constrained_layout=True)
top, bot = fig.subfigures(2, 1, height_ratios=[1.3, 1.25])   # bigger bottom block (d, e)

gs_top = top.add_gridspec(2, 2, width_ratios=[0.82, 1.55], height_ratios=[1.45, 0.95])
ax_a = top.add_subplot(gs_top[:, 0])
ax_b = top.add_subplot(gs_top[0, 1])
ax_c = top.add_subplot(gs_top[1, 1])

gs_bot = bot.add_gridspec(2, 1, hspace=0.06)
ax_d = bot.add_subplot(gs_bot[0])
ax_e = bot.add_subplot(gs_bot[1], sharex=ax_d)

_panel_a_peptides(ax_a, pos_training)
_panel_b_donut(ax_b, pos_training)
_panel_c_species(ax_c, pos_training_species)
_panels_de_boxplots(ax_d, ax_e, pos_training)

set_y_headroom([ax_c, ax_d, ax_e])
add_panel_labels([ax_a, ax_b, ax_c, ax_d, ax_e])

# Editable SVG (working format) + raster PNG preview.
save_figure(fig, "dataset_overview", subdir="fig2", formats=("svg", "png"))
plt.show()